In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import Cityscapes
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)
from torchvision.transforms import functional as TF
from torchvision.transforms import ColorJitter, InterpolationMode, RandomApply
from PIL import Image
from tqdm import tqdm


# Pre-Configuration

In [ ]:
# Dataset
DATA_ROOT = "./data"
CACHE_ROOT = "./cache"

NUM_CLASSES = 19
IGNORE_INDEX = 255

# Input
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 1024

# Training
BATCH_SIZE = 2
TOTAL_EPOCHS = 30
# Validation at full resolution is expensive, so run it every few epochs only
VAL_INTERVAL = 2
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PREFETCH_FACTOR = 2
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("Device:", DEVICE)


Device: cuda


# Dataset Processing

In [ ]:
ID_TO_TRAIN_ID = np.full(256, IGNORE_INDEX, dtype=np.uint8)

for original_id, train_id in {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4,
    17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
    23: 10, 24: 11, 25: 12, 26: 13, 27: 14,
    28: 15, 31: 16, 32: 17, 33: 18
}.items():
    ID_TO_TRAIN_ID[original_id] = train_id


def build_cache(split):
    """Resize + remap Avoid decode on each Epoch"""

    image_dir = Path(CACHE_ROOT) / split / "images"
    mask_dir = Path(CACHE_ROOT) / split / "masks"

    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    source = Cityscapes(
        root=DATA_ROOT,
        split=split,
        mode="fine",
        target_type="semantic"
    )

    for index in tqdm(range(len(source)), desc=f"Caching {split}"):

        image_path = image_dir / f"{index:05d}.jpg"
        mask_path = mask_dir / f"{index:05d}.png"

        if image_path.exists() and mask_path.exists():
            continue

        image, mask = source[index]

        image.resize(
            (IMAGE_WIDTH, IMAGE_HEIGHT),
            Image.BILINEAR
        ).save(image_path, quality=95)

        mask = mask.resize(
            (IMAGE_WIDTH, IMAGE_HEIGHT),
            Image.NEAREST
        )

        Image.fromarray(
            ID_TO_TRAIN_ID[np.array(mask, dtype=np.uint8)]
        ).save(mask_path)


build_cache("train")
build_cache("val")


Caching val: 100%|██████████| 500/500 [00:00<00:00, 3457.72it/s]


In [ ]:
CLASS_NAMES = [
    "road",
    "sidewalk",
    "building",
    "wall",
    "fence",
    "pole",
    "traffic light",
    "traffic sign",
    "vegetation",
    "terrain",
    "sky",
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motorcycle",
    "bicycle"
]


class CityscapesDataset(torch.utils.data.Dataset):

    def __init__(self, split="train", augment=False):

        self.image_dir = Path(CACHE_ROOT) / split / "images"
        self.mask_dir = Path(CACHE_ROOT) / split / "masks"

        self.names = sorted(
            path.stem for path in self.image_dir.glob("*.jpg")
        )

        if not self.names:
            raise RuntimeError(
                f"Empty cache for split '{split}'. Kindly run build cache firstly('{split}')."
            )

        self.augment = augment

        self.color_jitter = RandomApply(
            [
                ColorJitter(
                    brightness=0.2,
                    contrast=0.2,
                    saturation=0.2
                )
            ],
            p=0.5
        )

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):

        name = self.names[index]

        image = Image.open(
            self.image_dir / f"{name}.jpg"
        ).convert("RGB")

        mask = Image.open(
            self.mask_dir / f"{name}.png"
        )

        if self.augment:

            if torch.rand(1).item() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

            image = self.color_jitter(image)

        # ImageNet normalization
        image = TF.to_tensor(image)

        image = TF.normalize(
            image,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        mask = torch.from_numpy(np.array(mask, dtype=np.uint8))

        return image, mask


# MobileNetV3 - Encoder

In [6]:
class MobileNetV3Encoder(nn.Module):

    def __init__(self, pretrained=True):

        super().__init__()

        weights = (
            MobileNet_V3_Large_Weights.DEFAULT
            if pretrained else None
        )

        self.backbone = mobilenet_v3_large(
            weights=weights
        )

        features = self.backbone.features

        # Change output stride from 32 to 16
        # Block 13 originally uses stride=2
        for idx in [13, 14, 15]:

            block = features[idx]

            # Depthwise convolution
            depthwise = block.block[1][0]

            kernel_size = depthwise.kernel_size[0]

            depthwise.stride = (1, 1)

            depthwise.dilation = (2, 2)

            padding = 2 * (kernel_size - 1) // 2

            depthwise.padding = (
                padding,
                padding
            )

        self.features = features

    def forward(self, x):

        low_level = None

        for idx, layer in enumerate(self.features):

            x = layer(x)

            if idx == 2:
                low_level = x

        high_level = x

        return low_level, high_level

# ASPP - Atrous Spatial Pyramid Pooling

In [7]:
class ASPP(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels=256
    ):

        super().__init__()

        def conv_block(
            in_ch,
            out_ch,
            kernel_size,
            padding=0,
            dilation=1
        ):

            return nn.Sequential(

                nn.Conv2d(
                    in_ch,
                    out_ch,
                    kernel_size,
                    padding=padding,
                    dilation=dilation,
                    bias=False
                ),

                nn.BatchNorm2d(out_ch),

                nn.ReLU(inplace=True)
            )

        self.branch1 = conv_block(
            in_channels,
            out_channels,
            1
        )

        self.branch2 = conv_block(
            in_channels,
            out_channels,
            3,
            padding=6,
            dilation=6
        )

        self.branch3 = conv_block(
            in_channels,
            out_channels,
            3,
            padding=12,
            dilation=12
        )

        self.branch4 = conv_block(
            in_channels,
            out_channels,
            3,
            padding=18,
            dilation=18
        )

        self.global_pool = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True)
        )

        self.project = nn.Sequential(

            nn.Conv2d(
                out_channels * 5,
                out_channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True),

            nn.Dropout(0.1)
        )

    def forward(self, x):

        size = x.shape[-2:]

        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)

        b5 = self.global_pool(x)

        b5 = F.interpolate(
            b5,
            size=size,
            mode="bilinear",
            align_corners=False
        )

        out = torch.cat(
            [b1, b2, b3, b4, b5],
            dim=1
        )

        return self.project(out)

# DeepLabV3 - Decoder

In [8]:
class DeepLabV3Plus(nn.Module):

    def __init__(
        self,
        num_classes=19,
        pretrained=True
    ):

        super().__init__()

        self.encoder = MobileNetV3Encoder(
            pretrained=pretrained
        )

        self.aspp = ASPP(
            in_channels=960,
            out_channels=256
        )

        # Low-level feature projection
        self.low_level_projection = nn.Sequential(

            nn.Conv2d(
                24,
                48,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(48),

            nn.ReLU(inplace=True)
        )

        # Decoder
        self.decoder = nn.Sequential(

            nn.Conv2d(
                256 + 48,
                256,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                256,
                256,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(inplace=True)
        )

        self.classifier = nn.Conv2d(
            256,
            num_classes,
            kernel_size=1
        )

    def forward(self, x):

        input_size = x.shape[-2:]

        low_level, high_level = self.encoder(x)

        # ASPP
        high_level = self.aspp(high_level)

        # Upsample high-level features
        high_level = F.interpolate(
            high_level,
            size=low_level.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        # Project low-level features
        low_level = self.low_level_projection(
            low_level
        )

        # Feature fusion
        x = torch.cat(
            [high_level, low_level],
            dim=1
        )

        # Decoder
        x = self.decoder(x)

        # Classification
        x = self.classifier(x)

        # Upsample to input resolution
        x = F.interpolate(
            x,
            size=input_size,
            mode="bilinear",
            align_corners=False
        )

        return x

In [9]:
model = DeepLabV3Plus(
    num_classes=NUM_CLASSES,
    pretrained=True
).to(DEVICE, memory_format=torch.channels_last)

model.eval()

dummy = torch.randn(
    2, 3, IMAGE_HEIGHT, IMAGE_WIDTH
).to(DEVICE, memory_format=torch.channels_last)

with torch.no_grad():
    output = model(dummy)

print(output.shape)


torch.Size([2, 19, 512, 1024])


# Training and Evaluation

## Loss Function

In [10]:
criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX
)

# Confusion Matrix

In [11]:
class SegmentationMetrics:

    def __init__(self, num_classes, device):

        self.num_classes = num_classes

        self.confusion_matrix = torch.zeros(
            (num_classes, num_classes),
            dtype=torch.int64,
            device=device
        )

    @torch.no_grad()
    def update(self, predictions, targets):

        predictions = predictions.argmax(dim=1)

        valid = targets != IGNORE_INDEX

        targets = targets[valid].long()
        predictions = predictions[valid].long()

        indices = (
            targets * self.num_classes + predictions
        )

        matrix = torch.bincount(
            indices,
            minlength=self.num_classes ** 2
        )

        self.confusion_matrix += matrix.reshape(
            self.num_classes,
            self.num_classes
        )

    def compute(self):

        cm = self.confusion_matrix.float()

        tp = torch.diag(cm)

        fp = cm.sum(dim=0) - tp

        fn = cm.sum(dim=1) - tp

        # IoU
        union = tp + fp + fn

        iou = tp / union.clamp(min=1e-8)

        valid_iou = union > 0

        miou = iou[valid_iou].mean()

        # Precision
        precision = tp / (tp + fp).clamp(min=1e-8)

        # Recall
        recall = tp / (tp + fn).clamp(min=1e-8)

        # F1-score
        f1 = (
            2 * precision * recall /
            (precision + recall).clamp(min=1e-8)
        )

        # Pixel accuracy
        pixel_acc = (
            tp.sum() / cm.sum().clamp(min=1e-8)
        )

        return {
            "pixel_accuracy": pixel_acc.item(),
            "mIoU": miou.item(),
            "precision": precision.mean().item(),
            "recall": recall.mean().item(),
            "f1_score": f1.mean().item(),
            "class_iou": iou.cpu().numpy()
        }

## Training

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device
):
    model.train()

    total_loss = torch.zeros((), device=device)
    correct_pixels = torch.zeros((), device=device)
    total_pixels = torch.zeros((), device=device)
    num_samples = 0

    progress = tqdm(loader, desc="Training")

    for step, (images, masks) in enumerate(progress):

        images = images.to(
            device,
            non_blocking=True
        ).contiguous(memory_format=torch.channels_last)

        masks = masks.to(
            device,
            dtype=torch.long,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.detach() * images.size(0)
        num_samples += images.size(0)

        with torch.no_grad():
            predictions = outputs.detach().argmax(dim=1)
            valid = masks != IGNORE_INDEX
            correct_pixels += ((predictions == masks) & valid).sum()
            total_pixels += valid.sum()

        if step % 50 == 0:
            progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = (total_loss / num_samples).item()
    accuracy = (correct_pixels.float() / total_pixels.clamp(min=1)).item()

    return avg_loss, accuracy

In [13]:
def calculate_metrics(confusion_matrix):

    cm = confusion_matrix.float()

    # True Positive
    TP = torch.diag(cm)

    # False Positive
    FP = cm.sum(dim=0) - TP

    # False Negative
    FN = cm.sum(dim=1) - TP

    # IoU
    iou = TP / (TP + FP + FN).clamp(min=1e-7)

    # Precision
    precision = TP / (TP + FP).clamp(min=1e-7)

    # Recall
    recall = TP / (TP + FN).clamp(min=1e-7)

    # F1-score
    f1 = (
        2 * precision * recall
        / (precision + recall).clamp(min=1e-7)
    )

    # Pixel Accuracy
    pixel_accuracy = (
        TP.sum() / cm.sum().clamp(min=1e-7)
    )

    # Mean IoU
    present_classes = (TP + FP + FN) > 0

    miou = iou[present_classes].mean()

    # Macro Precision / Recall / F1
    precision_mean = precision[present_classes].mean()
    recall_mean = recall[present_classes].mean()
    f1_mean = f1[present_classes].mean()

    return {
        "pixel_accuracy": pixel_accuracy.item(),
        "mIoU": miou.item(),
        "precision": precision_mean.item(),
        "recall": recall_mean.item(),
        "f1_score": f1_mean.item(),
        "class_iou": iou.cpu().numpy()
    }

## Validation

In [ ]:
def validate(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    num_samples = 0

    confusion_matrix = torch.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        device=device,
        dtype=torch.int64
    )

    progress = tqdm(loader, desc="Validation")

    with torch.no_grad():
        for images, masks in progress:
            images = images.to(
                device,
                non_blocking=True
            ).contiguous(memory_format=torch.channels_last)

            masks = masks.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, masks)

            total_loss += loss.item() * images.size(0)
            num_samples += images.size(0)

            predictions = outputs.argmax(dim=1)
            valid = masks != IGNORE_INDEX
            
            target_pixels = masks[valid]
            predicted_pixels = predictions[valid]

            indices = target_pixels * NUM_CLASSES + predicted_pixels

            confusion_matrix += torch.bincount(
                indices,
                minlength=NUM_CLASSES * NUM_CLASSES
            ).reshape(NUM_CLASSES, NUM_CLASSES)

    avg_loss = total_loss / num_samples
    metrics = calculate_metrics(confusion_matrix)

    return avg_loss, metrics

## Main Training

In [ ]:
def main():

    # Dataset
    train_dataset = CityscapesDataset(
        split="train",
        augment=True
    )

    val_dataset = CityscapesDataset(
        split="val",
        augment=False
    )

    # DataLoader
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    # Model
    model = DeepLabV3Plus(
        num_classes=NUM_CLASSES,
        pretrained=True
    ).to(DEVICE, memory_format=torch.channels_last)

    # Loss
    criterion = nn.CrossEntropyLoss(
        ignore_index=IGNORE_INDEX
    )

    # Different learning rates for backbone and decoder
    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.encoder.parameters(),
                "lr": BACKBONE_LR
            },
            {
                "params": model.aspp.parameters(),
                "lr": HEAD_LR
            },
            {
                "params": model.low_level_projection.parameters(),
                "lr": HEAD_LR
            },
            {
                "params": model.decoder.parameters(),
                "lr": HEAD_LR
            },
            {
                "params": model.classifier.parameters(),
                "lr": HEAD_LR
            }
        ],
        weight_decay=WEIGHT_DECAY
    )

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5
    )

    # Mixed precision
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )

    os.makedirs(
        "checkpoints",
        exist_ok=True
    )

    best_miou = 0.0

    checkpoint_path = "checkpoints/latest_model.pth"
    start_epoch = 0
    best_miou = 0.0

    if os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        if "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if "scaler_state_dict" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])
            
        start_epoch = checkpoint["epoch"]
        best_miou = checkpoint.get("best_miou", 0.0)
        
        print(f"Resumed successfully from epoch {start_epoch} | Best mIoU: {best_miou:.4f}")
    
    for epoch in range(start_epoch, TOTAL_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{TOTAL_EPOCHS}")
        # Training
        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler,
            DEVICE
        )

        # Validation
        val_loss, val_metrics = validate(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        scheduler.step(
            val_metrics["mIoU"]
        )

        current_lr_backbone = optimizer.param_groups[0]["lr"]
        current_lr_head = optimizer.param_groups[1]["lr"]

        print("\nTraining Results")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Pixel Accuracy: {train_accuracy:.4f}")

        print("\nValidation Results")
        print(f"Val Loss: {val_loss:.4f}")

        print(
            f"Pixel Accuracy: "
            f"{val_metrics['pixel_accuracy']:.4f}"
        )

        print(
            f"mIoU: {val_metrics['mIoU']:.4f}"
        )

        print(
            f"Precision: {val_metrics['precision']:.4f}"
        )

        print(
            f"Recall: {val_metrics['recall']:.4f}"
        )

        print(
            f"F1-score: {val_metrics['f1_score']:.4f}"
        )

        print(
            f"Backbone LR: {current_lr_backbone:.6f}"
        )

        print(
            f"Head LR: {current_lr_head:.6f}"
        )

        # Save best model
        if val_metrics["mIoU"] > best_miou:
            best_miou = val_metrics["mIoU"]
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(), # Thêm dòng này
                    "scaler_state_dict": scaler.state_dict(),       # Thêm dòng này
                    "best_miou": best_miou,
                    "class_names": CLASS_NAMES
                },
                "checkpoints/best_model.pth"
            )
            print("Best model saved!")

        # Save latest checkpoint
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(), # Thêm dòng này
                "scaler_state_dict": scaler.state_dict(),       # Thêm dòng này
                "best_miou": best_miou
            },
            "checkpoints/latest_model.pth"
        )

    print("\nTraining completed!")
    print(f"Best mIoU: {best_miou:.4f}")


if __name__ == "__main__":
    main()


Epoch 1/30


Validation: 100%|██████████| 250/250 [00:36<00:00,  6.92it/s]



Training Results
Train Loss: 0.4057
Train Pixel Accuracy: 0.8766

Validation Results
Val Loss: 0.3082
Pixel Accuracy: 0.9001
mIoU: 0.4446
Precision: 0.7179
Recall: 0.5303
F1-score: 0.5531
Backbone LR: 0.000100
Head LR: 0.001000
Best model saved!

Epoch 2/30


Validation: 100%|██████████| 250/250 [00:36<00:00,  6.78it/s]



Training Results
Train Loss: 0.2655
Train Pixel Accuracy: 0.9144

Validation Results
Val Loss: 0.2585
Pixel Accuracy: 0.9156
mIoU: 0.5150
Precision: 0.7075
Recall: 0.6181
F1-score: 0.6352
Backbone LR: 0.000100
Head LR: 0.001000
Best model saved!

Epoch 3/30


Validation: 100%|██████████| 250/250 [00:42<00:00,  5.86it/s]


... (51 outputs left out)

Validation: 100%|██████████| 250/250 [00:09<00:00, 26.35it/s]



Training Results
Train Loss: 0.0614
Train Pixel Accuracy: 0.9768

Validation Results
Val Loss: 0.2586
Pixel Accuracy: 0.9425
mIoU: 0.6609
Precision: 0.8243
Recall: 0.7493
F1-score: 0.7799
Backbone LR: 0.000025
Head LR: 0.000250

Epoch 30/30


Validation: 100%|██████████| 250/250 [00:09<00:00, 26.02it/s]



Training Results
Train Loss: 0.0603
Train Pixel Accuracy: 0.9772

Validation Results
Val Loss: 0.2680
Pixel Accuracy: 0.9423
mIoU: 0.6618
Precision: 0.8258
Recall: 0.7488
F1-score: 0.7811
Backbone LR: 0.000025
Head LR: 0.000250

Training completed!
Best mIoU: 0.6637


# Inference – Semantic Segmentation

In [16]:
def load_model(checkpoint_path, device):

    model = DeepLabV3Plus(
        num_classes=NUM_CLASSES,
        pretrained=False
    ).to(device)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    return model


@torch.no_grad()
def predict(model, image_path, device):

    image = Image.open(image_path).convert("RGB")

    original_size = image.size

    image = TF.resize(
        image,
        [IMAGE_HEIGHT, IMAGE_WIDTH],
        interpolation=InterpolationMode.BILINEAR
    )

    image = TF.to_tensor(image)

    image = TF.normalize(
        image,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

    image = image.unsqueeze(0).to(device)

    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):

        output = model(image)

    prediction = output.argmax(dim=1)

    prediction = prediction.squeeze(0).cpu().numpy()

    prediction = Image.fromarray(
        prediction.astype(np.uint8)
    )

    prediction = prediction.resize(
        original_size,
        resample=Image.Resampling.NEAREST
    )

    return prediction

In [17]:
CITYSCAPES_COLORS = [
    (128, 64, 128),   # Road
    (244, 35, 232),   # Sidewalk
    (70, 70, 70),     # Building
    (102, 102, 156),  # Wall
    (190, 153, 153),  # Fence
    (153, 153, 153),  # Pole
    (250, 170, 30),   # Traffic light
    (220, 220, 0),    # Traffic sign
    (107, 142, 35),   # Vegetation
    (152, 251, 152),  # Terrain
    (70, 130, 180),   # Sky
    (220, 20, 60),    # Person
    (255, 0, 0),      # Rider
    (0, 0, 142),      # Car
    (0, 0, 70),       # Truck
    (0, 60, 100),     # Bus
    (0, 80, 100),     # Train
    (0, 0, 230),      # Motorcycle
    (119, 11, 32)     # Bicycle
]


def colorize_mask(mask):

    mask = np.array(mask)

    color_mask = np.zeros(
        (*mask.shape, 3),
        dtype=np.uint8
    )

    for class_id, color in enumerate(CITYSCAPES_COLORS):

        color_mask[mask == class_id] = color

    return Image.fromarray(color_mask)

In [18]:
model = load_model(
    "checkpoints/best_model.pth",
    DEVICE
)

image_path = "C:\\Users\\TDA5HC\\Desktop\\Documents\\01_CS_Master\\12_DeepLearningAndApplicationInComputerVision\\DeepLearningAndComputerVision\\src\\A2\\data\\leftImg8bit\\val\\frankfurt\\frankfurt_000000_000576_leftImg8bit.png"

prediction = predict(
    model,
    image_path,
    DEVICE
)

color_mask = colorize_mask(prediction)

color_mask.save("segmentation_result.png")